# E.D.A. (Exploratory Data Analysis)

This exploratory analysis reviews the dataset's structure, sample records, data types, descriptive statistics, missing values, and formatting issues. The results establish a baseline for identifying data-quality problems and measuring the improvements made during cleaning.

In [ ]:
#packages to be imported
import pandas as pd 
import re #regex
import numpy as np #used for 

#Readability; moving path to a variable
path = "../Data/messy_Dataset_before.xlsx"
# Using pandas to read the excel file and store it in a dataframe
df = pd.read_excel(path)
#creating a copy of the original dataset to compare before and after cleaning
df_before = df.copy()
#Total rows before cleaning
messy_rows = len(df)
print(f"Rows before cleaning:{messy_rows}")
#displays the first 5 rows of the dataset
df.head()


In [ ]:
#Last 5 rows of the dataset
df.tail()

In [ ]:
#EDA
print(df.shape)# 16 columns 180 rows
df.info()
df.describe() #statistics

Using the `info()` method, we can identify several data quality issues within the dataset. Multiple columns contain incorrect or inconsistent data types that will need to be corrected before further analysis. Additionally, missing values are present in 10 of the columns.

A manual review of the original Excel file also revealed several formatting inconsistencies that may affect data quality and usability. These issues will need to be addressed during the cleaning process to ensure the dataset is accurate, consistent, and ready for analysis.


Before any data cleaning is performed, all identified data quality issues will be documented and quantified. This establishes a baseline that can be used to measure and demonstrate the changes made throughout the cleaning process.

The cleaning process will begin by standardizing the column names to ensure consistency and readability. Next, unnecessary spaces within string-based columns will be removed to improve data consistency and prevent formatting-related discrepancies.

In [ ]:
# Here we are removing leading and trailing whitespaces, lower casing all names, 
# and replacing any remaining whitespaces with underscores
df.columns = df.columns.str.strip().str.lower().str.replace(" ","_")

#loop through columns 
for columns in df.columns:
    #only use columns with string data types
    if df[columns].dtype == "str":
        #get ride of leading and trailing whitespaces
        df[columns] = df[columns].str.strip()

Several phone number entries are inconsistently formatted, including the use of parentheses and the `+1` country code. These inconsistencies will be removed to standardize the phone number format across the dataset.


In [ ]:
#cleaning phone column
#using regex to get ride of country code
df["phone"] = df["phone"].str.replace( r"^\+?1[\s.-]*", "", regex=True)

#removing anything non-numeric characters
df["phone"] = df["phone"].str.replace( r"\D", "", regex=True)

#reformatting for valid 10 digit number
df["phone"] = df["phone"].str.replace( r"^(\d{3})(\d{3})(\d{4})$", r"\1-\2-\3", regex=True)

The order_date column contains both date and time information. Because the analysis focuses on order dates rather than transaction times, the time component was removed and the field was standardized to a date format.Then I converted the columns to a datetime data type and removed the time component, retaining only the date.

In [ ]:
# Convert the 'order_date' column to datetime
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
#removing the time portion of the datetime column
df["order_date"] = df["order_date"].dt.date

Next, I will standardize the text formatting in the `city`, `product`, `category`, `sales_rep`, `customer_type`, and `order_status` columns. I will use Python’s `.capitalize()` and `.title()` string methods to ensure consistent capitalization throughout these fields.

In [ ]:
#title format
df["city"] = df["city"].str.title()
df["product"] = df["product"].str.title()
df["sales_rep"] = df["sales_rep"].str.title()
#capitalize format
df["customer_type"] = df["customer_type"].str.capitalize()
df["category"] = df["category"].str.capitalize()
df["order_status"] = df["order_status"].str.capitalize()

During the initial data-quality review, I identified negative values in the `quantity` column. Because quantity represents the number of items associated with an order and the dataset contained no indication that negative quantities represented returns, cancellations, or inventory adjustments, these values were treated as invalid.

In [ ]:
#using absolute value to eliminate unnecessary negatives
df["quantity"] = df["quantity"].abs()
df["quantity"] = df["quantity"].astype("Int64")

Next, I will standardize the `unit_price` column by removing currency symbols, currency codes, and other non-numeric characters to ensure the values are stored consistently as numerical data.

In [ ]:
#removing anything that isn't a number or decimal
df["unit_price"] = df["unit_price"].replace(r"[^0-9.]", "", regex=True)
#then I will change the data type to numeric
df["unit_price"] = pd.to_numeric( df["unit_price"],errors="coerce")

The categorical fields were standardized to ensure consistent formatting and terminology throughout the dataset. In the `order_status` column, **“cancelled”** was standardized to **“cancel”**, while **“completed”** was standardized to **“complete.”**

The `payment_method` column was standardized by converting **“cc”** to **“Credit Card”**, **“debit”** to **“Debit Card”**, and removing the unnecessary space from **“paypal.”**

The `customer_type` column was also standardized by converting **“N”** to **“New.”** Values recorded as **“Y”** were replaced with **“N/A”** to provide a consistent placeholder for values that are not applicable or available.

In [ ]:
#standardizing inputs in order_status
df["order_status"] = df["order_status"].replace("cancelled", "cancel")
df["order_status"] = df["order_status"].replace("completed", "complete")
#standardizing inputs in payment_method
df["payment_method"] = df["payment_method"].replace("CC", "Credit Card")
df["payment_method"] = df["payment_method"].replace("Debit", "Debit Card")
#removing all whitespace in payment_method
df["payment_method"] = df["payment_method"].str.replace(r"\s+", "",regex = True)
#standardizing inputs in customer_type
df["customer_type"] = df["customer_type"].replace("N", "New")
df["customer_type"] = df["customer_type"].replace("Y", "N/A")
#correcting computer
df["category"] = df["category"].replace("Computers","Computer")

After reviewing the `unit_price` column in Excel, two data-entry issues were identified: one record contained an invalid value of **9999.99**, while another record contained a blank value. Both values will be converted to `NaN` to indicate missing or unavailable information, allowing the client to provide the appropriate values. These changes will be documented as part of the data-cleaning process.

In [ ]:
#Adjusting inputs for the client that seem like errors
df["unit_price"] = df["unit_price"].replace(" ", np.nan)
df["unit_price"] = df["unit_price"].replace(9999.99, np.nan)

In [ ]:
#counting missing values
missing_values = df.isna().sum()

print(missing_values)

#displaying rows that have missing values
missing_row = df[df.isna().any(axis =1)]
missing_row

When a customer name was missing but an email address was available, the username portion of the email address was used to generate a potential customer name. This approach was applied only when the available email information provided a reasonable basis for identifying the customer.

For fields where the missing information could not be reliably determined, the value will be replaced with `N/A` to indicate that the information is unknown. This applies to the `city`, `phone`, `email`,`sales_rep` and `payment_method` columns.

 

In [ ]:
#using regex to get name from email
name = df["customer_name"].isna() & df["email"].notna()

df.loc[name, "customer_name"] = (
    df.loc[name, "email"]
      .str.split("@")
      .str[0]
      .str.replace(r"[._-]+", " ", regex=True)
      .str.title()
)

#Filling in N/A for unknown values in the dataset
for col in ["city", "phone", "payment_method","email","sales_rep"]:
    df[col] = df[col].fillna("N/A")

#checking remaining rows with missing values
count = df.isna().sum()
print(count)

After reviewing the remaining rows with missing data, I decided to leave the values unchanged. Assigning replacement values without sufficient information could misrepresent the underlying data and potentially lead to issues for the client. These rows will be documented so the client can review and manually adjust the values if appropriate.

In [ ]:
adDf_before = df_before.copy()
#fixing column names to compare
adDf_before.columns = (
    adDf_before.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
#checking new df to old unchanged df
changes = adDf_before != df
#summing changes
col_chg = changes.sum()
#Summary of changes
print(col_chg)

Duplicate customer IDs were identified and reviewed. Redundant records were removed to ensure each customer ID is represented by a single unique record.

In [ ]:
#removing duplicates
df = df.drop_duplicates(subset="customer_id")
#displays rows with with nan
df[df.isna().any(axis=1)]

Once all cleaning and validation steps have been completed, the finalized dataset will be exported to an Excel file and prepared for delivery to the client.

In [ ]:
df.to_excel("../Data/cleaned_data.xlsx", index=False)